# Cross-modal PCA/PLS closed-form overview

This notebook is an onboarding guide for the closed-form PCA/PLS family in `models/architectures/crossmodal_pca_pls.py`.

It covers three models:

1. `CrossModalPCA`: shared-index PCA transfer from source latent coordinates to target PCA reconstruction.
2. `CrossModal_PLS_SVD`: raw-edge PLS-SVD through an implicit source-target cross-covariance operator.
3. `CrossModal_PCA_PLS`: PCA compression on each modality followed by PLS regression in latent space.

The goal is conceptual clarity plus a local smoke test for each model through the same `Sim.run_single(...)` and evaluator path used elsewhere in the project.

In [ ]:
import importlib
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import display

import main
import models.registry
import models.eval.evaluator
import models.architectures.crossmodal_pca_pls
from models.architectures.utils import get_model_input

importlib.reload(models.architectures.crossmodal_pca_pls)
importlib.reload(models.eval.evaluator)
importlib.reload(models.registry)
importlib.reload(main)

from main import Sim

RESULTS_ROOT = Path("results/local_results/crossmodal_pca_pls_closed_form_overview")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

BASE_SIM_KWARGS = dict(
    source="SC",
    target="FC",
    parcellation="Glasser",
    shuffle_seed=0,
    data_load_mode="precomputed",
)

def show_base_metrics(run_out):
    metrics = run_out["test_metrics"]["base_metrics"]
    return pd.Series(metrics).sort_index()


def inspect_one_batch(sim, model, title):
    batch = next(iter(sim.val_loader))
    x = get_model_input(batch)
    y = batch["y"]
    model.eval()
    with torch.no_grad():
        y_hat = model(x)
    print(title)
    print("x shape:", tuple(x.shape) if torch.is_tensor(x) else {k: tuple(v.shape) for k, v in x.items()})
    print("y shape:", tuple(y.shape))
    print("y_hat shape:", tuple(y_hat.shape))
    print("one-batch mse:", float(torch.mean((y_hat.cpu() - y.cpu()) ** 2)))
    return y_hat, y


## Shared data path

Each local test below constructs a `Sim` object with `data_load_mode="precomputed"`. That gives the notebook the same train, validation, and test loaders used by the project while avoiding the slower raw-file loading path.

The important object chain is:

$$
\texttt{Sim} \rightarrow \texttt{HCP\_Base} \rightarrow
(\texttt{train\_loader}, \texttt{val\_loader}, \texttt{test\_loader})
\rightarrow \texttt{Evaluator}.
$$

Closed-form models do not run gradient training. `Sim.run_single(...)` builds the model from the train split statistics, then evaluates train, validation, and test splits.

In [ ]:
closed_form_loader_probe = Sim(
    model_name="CrossModalPCA",
    config_overrides={"model": {"num_components": 32, "device": "cpu"}, "trainer": {"batch_size": 64}},
    **BASE_SIM_KWARGS,
)

print("train batches:", len(closed_form_loader_probe.train_loader))
print("val batches:", len(closed_form_loader_probe.val_loader))
print("test batches:", len(closed_form_loader_probe.test_loader))
probe_batch = next(iter(closed_form_loader_probe.train_loader))
print("batch keys:", sorted(probe_batch.keys()))
print("x shape:", tuple(get_model_input(probe_batch).shape))
print("y shape:", tuple(probe_batch["y"].shape))


## 1. `CrossModalPCA`

This is the simplest latent-space baseline. It assumes that source and target PCA components are aligned by component index. For an input source edge vector $x_s$, the model centers by the source training mean, projects into the first $k$ source PCA loadings, and decodes those same coordinates through the target PCA basis:

$$
z_s = (x_s - \mu_s) P_{s,k}
$$

$$
\hat y_t = \mu_t + z_s P_{t,k}^T.
$$

There is no learned source-to-target regression matrix. The only transformation between modalities is the decision to reuse the first $k$ source PCA coordinates as target PCA coordinates.

In [ ]:
pca_config = {
    "model": {
        "num_components": 64,
        "device": "cpu",
    },
    "trainer": {"batch_size": 64},
}

pca_sim = Sim(
    model_name="CrossModalPCA",
    config_overrides=pca_config,
    **BASE_SIM_KWARGS,
)

pca_run = pca_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=pca_config,
    run_eval=True,
)

show_base_metrics(pca_run)


In [ ]:
pca_y_hat, pca_y = inspect_one_batch(pca_sim, pca_run["model"], "CrossModalPCA one-batch check")


## 2. `CrossModal_PLS_SVD`

This model works in raw edge space and uses a memory-efficient SVD of the source-target cross-covariance. With centered train matrices $X$ and $Y$, it defines the cross-covariance operator

$$
C = X^T Y.
$$

Instead of materializing the full edge-by-edge matrix $C$, the implementation provides matrix-vector products:

$$
C v = X^T(Yv), \qquad C^T u = Y^T(Xu).
$$

The top singular vectors give source and target PLS directions:

$$
C \approx W_x S W_y^T.
$$

Training samples are projected into source and target PLS coordinates, then a small latent regression matrix is fit by least squares:

$$
Z_x = X W_x, \qquad Z_y = Y W_y, \qquad
B = \arg\min_B \|Z_x B - Z_y\|_F^2.
$$

Prediction is

$$
\hat y_t = \mu_t + (x_s - \mu_s) W_x B W_y^T.
$$

This is still closed form: the regression is fit once during model construction and no PyTorch optimizer is used.

In [ ]:
pls_svd_config = {
    "model": {
        "n_components": 8,
        "device": "cpu",
    },
    "trainer": {"batch_size": 64},
}

pls_svd_sim = Sim(
    model_name="CrossModal_PLS_SVD",
    config_overrides=pls_svd_config,
    **BASE_SIM_KWARGS,
)

pls_svd_run = pls_svd_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=pls_svd_config,
    run_eval=True,
)

show_base_metrics(pls_svd_run)


In [ ]:
pls_svd_y_hat, pls_svd_y = inspect_one_batch(pls_svd_sim, pls_svd_run["model"], "CrossModal_PLS_SVD one-batch check")
print("stored singular values:", pls_svd_run["model"].singular_values.detach().cpu().numpy())


## 3. `CrossModal_PCA_PLS`

This is the main closed-form PCA+PLS bridge. It first compresses each source modality and the target into PCA score space. For one source modality:

$$
z_s = (x_s - \mu_s) P_{s,k_s}, \qquad
z_t = (y_t - \mu_t) P_{t,k_t}.
$$

For multiple source modalities, the source latent vectors are concatenated:

$$
z_{src} = [z_{s_1}; z_{s_2}; \ldots; z_{s_m}].
$$

A scikit-learn `PLSRegression` model is fit from source PCA scores to target PCA scores:

$$
\hat z_t = f_{PLS}(z_{src}).
$$

The predicted target latents are decoded through the fixed target PCA basis:

$$
\hat y_t = \mu_t + \hat z_t P_{t,k_t}^T.
$$

Conceptually, this model asks whether a low-dimensional source PCA representation contains a PLS-aligned subspace that predicts low-dimensional FC structure.

In [ ]:
pca_pls_config = {
    "model": {
        "n_components_pca": 64,
        "n_components_pca_target": 64,
        "n_components_pls": 8,
        "device": "cpu",
    },
    "trainer": {"batch_size": 64},
}

pca_pls_sim = Sim(
    model_name="CrossModal_PCA_PLS",
    config_overrides=pca_pls_config,
    **BASE_SIM_KWARGS,
)

pca_pls_run = pca_pls_sim.run_single(
    mode="dev",
    save_checkpoint=False,
    config_override=pca_pls_config,
    run_eval=True,
)

show_base_metrics(pca_pls_run)


In [ ]:
pca_pls_y_hat, pca_pls_y = inspect_one_batch(pca_pls_sim, pca_pls_run["model"], "CrossModal_PCA_PLS one-batch check")
print("PLS x rotations shape:", tuple(pca_pls_run["model"].x_rotations_.shape))
print("target loadings shape:", tuple(pca_pls_run["model"].target_loadings_k.shape))


## Compare the closed-form smoke-test results

This table is a quick sanity check, not a tuned comparison. The dimensions above are intentionally small so that new contributors can run the notebook locally.

In [ ]:
closed_form_summary = pd.DataFrame({
    "CrossModalPCA": show_base_metrics(pca_run),
    "CrossModal_PLS_SVD": show_base_metrics(pls_svd_run),
    "CrossModal_PCA_PLS": show_base_metrics(pca_pls_run),
}).T

closed_form_summary


In [ ]:
artifact_dir = RESULTS_ROOT
artifact_dir.mkdir(parents=True, exist_ok=True)
for name, run_out in {
    "CrossModalPCA": pca_run,
    "CrossModal_PLS_SVD": pls_svd_run,
    "CrossModal_PCA_PLS": pca_pls_run,
}.items():
    report_base = artifact_dir / f"{name}_eval_test"
    test_eval = run_out["evaluators"]["test"]
    test_eval.analyze_results(
        verbose=False,
        filepath=str(report_base),
        output_format="md",
        model_name=name,
    )
    print(f"wrote {report_base}.md")
